# The text edit tool

Claude has a built-in text editor tool. Built-in means **Claude already knows the schema**:
you send a short stub, Anthropic expands it into the full tool spec. **Running** the commands is
still your job: Claude only asks "view `main.py`", "replace this text", "create `test.py`",
and your code touches the files.

| Command | What it does |
|---|---|
| `view` | show a file with line numbers (or a range), or list a directory |
| `create` | create a new file |
| `str_replace` | replace text that matches exactly one location |
| `insert` | insert text after a line |
| `undo_edit` | revert the last edit (course class; dropped from the newest built-in version) |

Use it to let an app edit files: code assistants, config generators, document tools.

## Setup

`chat` gets a `max_tokens` argument: a whole new file goes into one tool call.

In [1]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
from text_editor import TextEditorTool, text_editor_schema

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None, max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

## The built-in stub

The course code. The stub depends on the model; the current version strings are in the
[text editor tool docs](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/text-editor-tool).

**Our provider (OpenRouter) rejects it** with `Unknown server-tool shorthand`. So this notebook sends
the expanded form instead: a regular tool with the same name and commands, `text_editor_schema` in
[`text_editor.py`](./text_editor.py). With a direct Anthropic key the stub works as is.

In [2]:
from anthropic import BadRequestError


def get_text_edit_schema(model):
    if model.startswith("claude-3"):
        return {"type": "text_editor_20250124", "name": "str_replace_editor"}
    else:
        return {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}


try:
    chat([{"role": "user", "content": "Open main.py"}], tools=[get_text_edit_schema(model)])
    print("The stub works with this provider")
except BadRequestError as e:
    print("Rejected:", e.message)

Rejected: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Invalid Anthropic Messages API request', 'error_type': 'invalid_request'}, 'request_id': None, 'metadata': {'raw': '[\n  {\n    "code": "custom",\n    "path": [\n      "tools",\n      0,\n      "type"\n    ],\n    "message": "Unknown server-tool shorthand"\n  }\n]'}}


## Our side: `TextEditorTool`

The course class, rewritten shorter in [`text_editor.py`](./text_editor.py). Every path is checked to stay
inside one root folder: Claude must not be able to read or write anything else.
The course checks this with `startswith`, which lets `../sandbox-evil` through when the root is
`/sandbox`; here it is `Path.is_relative_to`.

The demo works in a temporary folder with one file, `main.py`.

In [3]:
import tempfile
from pathlib import Path

sandbox = Path(tempfile.mkdtemp(prefix="sandbox"))
(sandbox / "main.py").write_text('def greeting():\n    return "Hello! How can I assist you today?"\n')
editor = TextEditorTool(sandbox)

print(editor.view("main.py"))
print(editor.str_replace("main.py", "Hello!", "Hi!"))
print(editor.view("main.py", [2, 2]))
print(editor.undo_edit("main.py"))
print(editor.view("main.py", [2, 2]))

for path in ["../secret.txt", f"../{sandbox.name}-evil/x.py"]:
    try:
        editor.view(path)
    except ValueError as e:
        print(e)

1: def greeting():
2:     return "Hello! How can I assist you today?"
3: 
Successfully replaced text at exactly one location.
2:     return "Hi! How can I assist you today?"
Successfully restored main.py
2:     return "Hello! How can I assist you today?"
Access denied: '../secret.txt' is outside the allowed directory
Access denied: '../sandboxzxi1coh_-evil/x.py' is outside the allowed directory


## The loop

The loop from lesson 06. `run_tool` routes on `command` inside the one tool. The printout shows the command
and the path, not the whole input: `create` carries the whole file.

In [4]:
import json

tools = [text_editor_schema]


def run_tool(tool_name, tool_input):
    if tool_name != "str_replace_based_edit_tool":
        raise ValueError(f"Unknown tool: {tool_name}")

    command = tool_input["command"]
    if command == "view":
        return editor.view(tool_input["path"], tool_input.get("view_range"))
    elif command == "create":
        return editor.create(tool_input["path"], tool_input["file_text"])
    elif command == "str_replace":
        return editor.str_replace(tool_input["path"], tool_input["old_str"], tool_input["new_str"])
    elif command == "insert":
        return editor.insert(tool_input["path"], tool_input["insert_line"], tool_input["new_str"])
    elif command == "undo_edit":
        return editor.undo_edit(tool_input["path"])
    else:
        raise ValueError(f"Unknown text editor command: {command}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:  # noqa: BLE001 - any tool error goes back to Claude
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        first_line = tool_result_block["content"].split("\\n")[0]
        print(f"  [{tool_request.input.get('command')} {tool_request.input.get('path')} -> {first_line}]")
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools, max_tokens=4000)
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages


def print_history(messages):
    for number, message in enumerate(messages, start=1):
        content = message["content"]
        if isinstance(content, str):
            summary = repr(content)
        else:
            parts = []
            for block in content:
                if isinstance(block, dict):
                    parts.append(block["type"] + (" (error)" if block.get("is_error") else ""))
                else:
                    parts.append(block.type)
            summary = ", ".join(parts)
        print(f"{number}. {message['role']:9} {summary}")

## Read a file

In [5]:
messages = []
add_user_message(messages, "Open the main.py file and summarize its contents.")
run_conversation(messages)
print_history(messages)

I'll open the main.py file for you.
  [view main.py -> "1: def greeting():]


## Summary of main.py

The **main.py** file is a simple Python script containing a single function:

- **`greeting()` function**: A parameterless function that returns a greeting message string: *"Hello! How can I assist you today?"*

This appears to be a basic greeting function, likely intended as a starting point or template for a larger application. The function doesn't take any arguments and simply returns a pre-defined welcome message.
1. user      'Open the main.py file and summarize its contents.'
2. assistant text, tool_use
3. user      tool_result
4. assistant text


## Edit a file and create another one

The course prompt.

In [6]:
messages = []
add_user_message(
    messages,
    "Open the main.py file and write out a function to calculate pi to the 5th decimal place. "
    "Then create a test.py to test your implementation of the pi function.",
)
run_conversation(messages)
print_history(messages)

I'll help you create a function to calculate pi and write tests for it. Let me start by checking if main.py exists, then create the functions.
  [view . -> "main.py"]


Now let me view the main.py file:
  [view main.py -> "1: def greeting():]


Now I'll add a function to calculate pi to the 5th decimal place using the Bailey–Borwein–Plouffe formula (a well-known series):
  [str_replace main.py -> "Successfully replaced text at exactly one location."]


Now let me create a test.py file to test the pi calculation function:
  [create test.py -> "Successfully created test.py"]


Perfect! Let me view both files to confirm they were created correctly:
  [view main.py -> "1: def greeting():]
  [view test.py -> "1: import unittest]


Perfect! I've successfully completed your request. Here's what I created:

## Summary

### **main.py** - Added `calculate_pi()` function
The function calculates pi to the 5th decimal place (or any specified precision) using the **Machin formula**:
- Formula: `pi/4 = 4*arctan(1/5) - arctan(1/239)`
- Uses the `decimal` module for high-precision arithmetic
- Takes an optional `decimal_places` parameter (defaults to 5)
- Returns pi as a float rounded to the specified precision

**Key features:**
- Uses Taylor series to calculate arctan values
- Sets high internal precision to avoid rounding errors
- Returns accurate results for various decimal place requirements

### **test.py** - Comprehensive test suite
Created a complete test suite with 9 test cases:

**For `calculate_pi()`:**
- Tests pi calculation at 5 decimal places (default) → 3.14159
- Tests pi calculation at 2, 3, and 10 decimal places
- Verifies the return type is float
- Checks that pi is in the expected range (3.0 - 3.2)
- Vali

What is in the folder now, and do Claude's tests pass?

In [7]:
import os
import subprocess
import sys

for file in sorted(sandbox.iterdir()):
    print(f"===== {file.name}\n{file.read_text()}")

result = subprocess.run(
    [sys.executable, "test.py"], cwd=sandbox, capture_output=True, text=True, check=False,
    env={**os.environ, "PYTHON_COLORS": "0"},
)
print("exit code:", result.returncode)
print(result.stdout[-1500:], result.stderr[-1500:])

===== main.py
def greeting():
    return "Hello! How can I assist you today?"


def calculate_pi(decimal_places=5):
    """
    Calculate pi to a specified number of decimal places using the Machin formula.
    
    The Machin formula: pi/4 = 4*arctan(1/5) - arctan(1/239)
    
    Args:
        decimal_places (int): Number of decimal places to calculate (default: 5)
    
    Returns:
        float: Pi calculated to the specified decimal places
    """
    from decimal import Decimal, getcontext
    
    # Set precision higher than needed to avoid rounding errors
    getcontext().prec = decimal_places + 10
    
    # Using the Machin formula: pi/4 = 4*arctan(1/5) - arctan(1/239)
    def arctan(x, num_terms):
        """Calculate arctan(x) using Taylor series"""
        x = Decimal(x)
        power = x
        result = power
        for n in range(1, num_terms):
            power *= -x * x
            result += power / (2 * n + 1)
        return result
    
    # Calculate with sufficien

In [8]:
import shutil

shutil.rmtree(sandbox)

Next lesson: the web search tool.